In [1]:
# generate reviews in a csv

import csv
import random
from pathlib import Path

random.seed(42)

output_file = "employee_reviews.csv"

sentiment_templates = {
    "Positive": [
        ("Salary is competitive and fair.", "Salary"),
        ("Great work culture with supportive teammates.", "Culture"),
        ("Modern technology stack and useful tools.", "Technology"),
        ("Strong learning opportunities and skill growth.", "Growth"),
        ("Helpful perks like flexible hours and leave.", "Perks"),
        ("Management is approachable and supportive.", "Management"),
        ("Good performance bonuses and benefits.", "Salary"),
        ("Excellent collaboration across teams.", "Culture"),
        ("Projects use modern cloud and automation tools.", "Technology"),
        ("Clear opportunities to learn and advance.", "Growth"),
    ],
    "Negative": [
        ("Salary is below market expectations.", "Salary"),
        ("Work culture feels stressful and inconsistent.", "Culture"),
        ("The technology stack is outdated and slow.", "Technology"),
        ("Career growth is limited and unclear.", "Growth"),
        ("Perks are minimal compared to other companies.", "Perks"),
        ("Management communication is weak.", "Management"),
        ("Raises are small and infrequent.", "Salary"),
        ("Team morale is low in some departments.", "Culture"),
        ("Legacy systems make daily work harder.", "Technology"),
        ("Promotion paths are not well defined.", "Growth"),
    ],
    "Neutral": [
        ("Salary is average for the industry.", "Salary"),
        ("Work culture varies by team.", "Culture"),
        ("The technology stack is a mix of old and new tools.", "Technology"),
        ("Growth opportunities depend on the project.", "Growth"),
        ("Perks are standard and adequate.", "Perks"),
        ("Management style differs across departments.", "Management"),
        ("Compensation is acceptable for the role.", "Salary"),
        ("Work-life balance is fine most weeks.", "Culture"),
        ("Some tools are modern while others are legacy.", "Technology"),
        ("Learning is available, but mostly self-driven.", "Growth"),
    ],
}

extra_phrases = [
    "Overall", "In my experience", "For the most part", "At times", "From my team’s perspective",
    "Compared with peers", "In some cases", "Generally speaking"
]

rows = []
for i in range(1, 1001):
    sentiment = random.choices(["Positive", "Negative", "Neutral"], weights=[0.4, 0.3, 0.3])[0]
    template, topic = random.choice(sentiment_templates[sentiment])
    prefix = random.choice(extra_phrases)
    review = f"{prefix}, {template}"
    rows.append({
        "id": i,
        "review": review,
        "sentiment": sentiment,
        "topic": topic
    })

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "review", "sentiment", "topic"])
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} reviews to {output_file}")

Saved 1000 reviews to employee_reviews.csv


In [3]:
import pandas as pd

df = pd.read_csv("employee_reviews.csv")

df.head()

,id,review,sentiment,topic
0,1,"From my team’s perspective, Salary is below ma...",Negative,Salary
1,2,"In my experience, Modern technology stack and ...",Positive,Technology
2,3,"In my experience, Legacy systems make daily wo...",Negative,Technology
3,4,"Overall, Salary is below market expectations.",Negative,Salary
4,5,"Overall, Strong learning opportunities and ski...",Positive,Growth


In [32]:
#df["final_review"] = df["topic"] + " : " + df["review"]
df["final_review"] = df["review"]
df[["final_review","sentiment"]].head()

,final_review,sentiment
0,"From my team’s perspective, Salary is below ma...",Negative
1,"In my experience, Modern technology stack and ...",Positive
2,"In my experience, Legacy systems make daily wo...",Negative
3,"Overall, Salary is below market expectations.",Negative
4,"Overall, Strong learning opportunities and ski...",Positive


In [33]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded. Embedding dimension: {sbert_model.get_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4571.52it/s]


Model loaded. Embedding dimension: 384


In [34]:
%%time

embeddings = sbert_model.encode(df["final_review"].tolist(), show_progress_bar=True)
print(f"Embeddings shape: {embeddings.shape}")

Batches: 100%|██████████| 32/32 [00:04<00:00,  7.25it/s]


Embeddings shape: (1000, 384)
CPU times: total: 8.58 s
Wall time: 4.43 s


In [35]:
print(f"Review: {df['final_review'].iloc[0][:80]}...")
print(f"\nEmbedding (first 20 values): {embeddings[0][:20]}")
print(f"\nMin: {embeddings[0].min():.4f}, Max: {embeddings[0].max():.4f}, Mean: {embeddings[0].mean():.4f}")

Review: From my team’s perspective, Salary is below market expectations....

Embedding (first 20 values): [ 0.00853242  0.0644874  -0.03114571  0.02262287 -0.04961051 -0.02874859
  0.0304747   0.06955957  0.08714759 -0.00270475 -0.03704865 -0.04575799
  0.02344749 -0.03496746  0.04177963 -0.06611042  0.03275447 -0.06759392
 -0.02577854 -0.03289006]

Min: -0.1520, Max: 0.1439, Mean: -0.0005


In [36]:
# validate if SBERT understands meaning

from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "From my team’s perspective, Salary is below market expectations",
    "From my team’s perspective, Learning is available, but mostly self-driven.",  # completely different topic
    "In some cases, salary is not acceptable for the role.",                       # same meaning, different words
]

demo_embeddings = sbert_model.encode(sentences)
sim_matrix = cosine_similarity(demo_embeddings)

print("Cosine Similarity Matrix:")
print(f"  sentence 1 vs sentence 3 (same topic):       {sim_matrix[0][2]:.4f}")
print(f"  sentence 1 vs sentence 2 (different topic): {sim_matrix[0][1]:.4f}")

Cosine Similarity Matrix:
  sentence 1 vs sentence 3 (same topic):       0.5178
  sentence 1 vs sentence 2 (different topic): 0.2095


In [37]:
from sklearn.model_selection import train_test_split

y = df["sentiment"]
x_train, x_test, y_train, y_test = train_test_split(embeddings, y, test_size=0.2, random_state=42)

print(f"Training set: {x_train.shape}")
print(f"Test set:     {x_test.shape}")

Training set: (800, 384)
Test set:     (200, 384)


In [38]:
# training the model - Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(x_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [39]:
rf_model.score(x_test, y_test)

1.0

In [40]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_rf = rf_model.predict(x_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Confusion Matrix:
[[64  0  0]
 [ 0 59  0]
 [ 0  0 77]]

Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      1.00      1.00        64
     Neutral       1.00      1.00      1.00        59
    Positive       1.00      1.00      1.00        77

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [41]:
def predict_review_sentiment(review, model=rf_model):
    """Predict the Sentiment for a new review and show confidence."""
    embedding = sbert_model.encode([review])
    prediction = model.predict(embedding)[0]
    probabilities = model.predict_proba(embedding)[0]
    confidence = max(probabilities)
    
    print(f"Review:     {review}")
    print(f"Sentiment:      {prediction}")
    print(f"Confidence: {confidence:.2%}")
    print(f"\nAll probabilities:")
    for queue, prob in sorted(zip(model.classes_, probabilities), key=lambda x: -x[1]):
        bar = "█" * int(prob * 40)
        print(f"  {queue:25s} {prob:.2%} {bar}")
    print()

In [49]:
predict_review_sentiment("Overall, Work culture feels very good.")

Review:     Overall, Work culture feels very good.
Sentiment:      Positive
Confidence: 37.00%

All probabilities:
  Positive                  37.00% ██████████████
  Negative                  35.00% ██████████████
  Neutral                   28.00% ███████████

